# Select Best Overall Model

Single notebook to choose the best model across trained candidates with comparable metrics.

Outputs:
- `v2_rfdetr_opt_v1/artifacts/benchmarks/overall_leaderboard.csv`
- `v2_rfdetr_opt_v1/artifacts/benchmarks/overall_champion.json`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List

import pandas as pd

In [ ]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v2_rfdetr_opt_v1').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root.')


REPO_ROOT = discover_repo_root()
BENCH_DIR = REPO_ROOT / 'v2_rfdetr_opt_v1' / 'artifacts' / 'benchmarks'
RUNS_DIR = REPO_ROOT / 'v2_rfdetr_opt_v1' / 'artifacts' / 'runs'
OVERALL_CSV = BENCH_DIR / 'overall_leaderboard.csv'
OVERALL_JSON = BENCH_DIR / 'overall_champion.json'

WEIGHTS = {
    'f1': 0.50,
    'mean_iou': 0.30,
    'area_consistency': 0.20,
    'speed': 0.00,
}

print('Repo root:', REPO_ROOT)
print('Bench dir:', BENCH_DIR)
print('Output CSV:', OVERALL_CSV)
print('Output JSON:', OVERALL_JSON)

In [ ]:
def parse_run_name(sweep_csv_path: Path) -> str:
    stem = sweep_csv_path.stem
    prefix = 'validation_conf_sweep_'
    return stem[len(prefix):] if stem.startswith(prefix) else stem


def locate_checkpoint(run_name: str) -> Path:
    run_dir = RUNS_DIR / run_name
    candidates = [
        run_dir / 'checkpoint_best_total.pth',
        run_dir / 'checkpoint_best_regular.pth',
        run_dir / 'checkpoint_best_ema.pth',
    ]
    for c in candidates:
        if c.is_file():
            return c
    raise FileNotFoundError(f'No best checkpoint found for run: {run_name}')


def load_speed_ips(run_name: str) -> float:
    p = BENCH_DIR / f'inference_benchmark_{run_name}.json'
    if not p.exists():
        return float('nan')
    data = json.loads(p.read_text(encoding='utf-8'))
    return float(data.get('images_per_sec', float('nan')))

In [ ]:
# Configuration is above. Run the next cell to build the overall leaderboard.

## Build Overall Leaderboard

In [ ]:
# Build unified candidate table across trained models with comparable metrics.
candidates = []

# RF-DETR candidates: best confidence row per trained sweep run.
rf_sweep_files = sorted(BENCH_DIR.glob('validation_conf_sweep_*.csv'))
for sweep_path in rf_sweep_files:
    run_name = parse_run_name(sweep_path)
    df = pd.read_csv(sweep_path)
    if df.empty:
        continue
    best = df.sort_values('score', ascending=False).iloc[0].to_dict()
    candidates.append(
        {
            'model_family': 'rfdetr_seg_small',
            'variant': run_name,
            'conf': float(best['conf']),
            'checkpoint': str(locate_checkpoint(run_name)),
            'f1': float(best['f1']),
            'mean_iou': float(best['mean_iou']),
            'area_rel_error': float(best['area_rel_error']),
            'images_per_sec': load_speed_ips(run_name),
        }
    )

# YOLO11s candidate: best confidence from YOLO sweep + baseline benchmark speed.
yolo_root = REPO_ROOT / 'v2_yolo11s_opt_v1' / 'artifacts'
yolo_conf_sweep = yolo_root / 'benchmarks' / 'conf_sweep.csv'
yolo_bench_base = yolo_root / 'benchmarks' / 'benchmark_baseline.json'
yolo_ckpt = REPO_ROOT / 'v2' / 'runs' / 'segment' / 'placentas_v11_aug_v2' / 'weights' / 'best.pt'

if yolo_conf_sweep.exists() and yolo_bench_base.exists() and yolo_ckpt.exists():
    ydf = pd.read_csv(yolo_conf_sweep).sort_values('score', ascending=False).reset_index(drop=True)
    ybest = ydf.iloc[0].to_dict()
    ybench = json.loads(yolo_bench_base.read_text(encoding='utf-8'))
    candidates.append(
        {
            'model_family': 'yolo11s_seg',
            'variant': 'v2_yolo11s_baseline',
            'conf': float(ybest['conf']),
            'checkpoint': str(yolo_ckpt),
            'f1': float(ybest['f1']),
            'mean_iou': float(ybest['mean_iou']),
            'area_rel_error': float(ybest['area_rel_error']),
            'images_per_sec': float(ybench.get('images_per_sec', float('nan'))),
        }
    )

overall = pd.DataFrame(candidates)
if overall.empty:
    raise RuntimeError('No comparable model candidates found.')

overall['area_consistency'] = 1.0 - overall['area_rel_error']

speed = overall['images_per_sec']
if speed.notna().sum() >= 2 and float(speed.max() - speed.min()) > 1e-12:
    overall['speed_norm'] = (speed - speed.min()) / (speed.max() - speed.min())
elif speed.notna().sum() == 1:
    overall['speed_norm'] = speed.notna().astype(float)
else:
    overall['speed_norm'] = 0.0

wsum = sum(WEIGHTS.values())
overall['weighted_score'] = (
    WEIGHTS['f1'] * overall['f1']
    + WEIGHTS['mean_iou'] * overall['mean_iou']
    + WEIGHTS['area_consistency'] * overall['area_consistency']
    + WEIGHTS['speed'] * overall['speed_norm']
) / wsum

overall = overall.sort_values('weighted_score', ascending=False).reset_index(drop=True)
overall.to_csv(OVERALL_CSV, index=False)

champ = overall.iloc[0].to_dict()
overall_champion = {
    'model_family': champ['model_family'],
    'variant': champ['variant'],
    'best_conf': float(champ['conf']),
    'checkpoint': champ['checkpoint'],
    'f1': float(champ['f1']),
    'mean_iou': float(champ['mean_iou']),
    'area_rel_error': float(champ['area_rel_error']),
    'images_per_sec': float(champ['images_per_sec']) if pd.notna(champ['images_per_sec']) else None,
    'weighted_score': float(champ['weighted_score']),
    'weights': WEIGHTS,
    'overall_leaderboard_csv': str(OVERALL_CSV),
}
OVERALL_JSON.write_text(json.dumps(overall_champion, indent=2), encoding='utf-8')

print('Saved:', OVERALL_CSV)
print('Saved:', OVERALL_JSON)
overall[['model_family','variant','conf','f1','mean_iou','area_rel_error','images_per_sec','weighted_score']]